# Context-Aware Dream Interpretation System

## Project Goal

Build a system that summarizes a dream and generates tentative, evidence-grounded theories about its themes. The system can retrieve related past dreams based on recurring people, locations, setting types, events, and emotions.

## Research Questions

1. Does QLoRA improve structured extraction of people, locations, settings, events, and emotions compared with an unchanged base model?

2. Does RAG make dream theories more personally relevant and grounded than analyzing the current dream alone, or can retrieval reduce quality?



In [13]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cpu
GPU available: False


In [15]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.2/927.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.2 MB/s eta 0:00:00


In [17]:
import random
import numpy as np
import torch

from datasets import Dataset
from transformers import set_seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("Setup complete.")

Setup complete.


## RAG Corpus: Load Private Dream Entries

In [18]:
import os

DATA_FOLDER = "/content/drive/MyDrive/IT344/DJournal Project/private_data"

print(os.listdir(DATA_FOLDER))

['dream2.txt', 'dream3.txt', 'dream4.txt', 'dream5.txt', 'dream6.txt', 'dream7.txt', 'dream8.txt', 'dream9.txt', 'dream10.txt', 'dream11.txt', 'dream12.txt', 'dream13.txt', 'dream14.txt', 'dream15.txt', '22.txt', '21.txt', '23.txt', '18.txt', '17.txt', '16.txt', '20.txt', '19.txt', '24.txt', 'Dream1.txt']


In [19]:
from pathlib import Path
import re

def dream_number(path):
    match = re.search(r"\d+", path.stem)
    return int(match.group()) if match else 999999

dream_files = sorted(
    Path(DATA_FOLDER).glob("*.txt"),
    key=dream_number
)

dream_entries = []

for file_path in dream_files:
    dream_entries.append({
        "dream_id": file_path.stem,
        "text": file_path.read_text(encoding="utf-8").strip()
    })

print("Dreams loaded:", len(dream_entries))
print("Files:", [dream["dream_id"] for dream in dream_entries])

Dreams loaded: 24
Files: ['Dream1', 'dream2', 'dream3', 'dream4', 'dream5', 'dream6', 'dream7', 'dream8', 'dream9', 'dream10', 'dream11', 'dream12', 'dream13', 'dream14', 'dream15', '16', '17', '18', '19', '20', '21', '22', '23', '24']


## Create Dream Embeddings


In [20]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

dream_texts = [dream["text"] for dream in dream_entries]

dream_embeddings = embedding_model.encode(
    dream_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", dream_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (24, 384)


## Build the Retrieval Index

In [21]:
import faiss

embedding_dimension = dream_embeddings.shape[1]

rag_index = faiss.IndexFlatIP(embedding_dimension)
rag_index.add(dream_embeddings.astype("float32"))

print("Dreams indexed:", rag_index.ntotal)

Dreams indexed: 24


## Dream Retrieval Function


In [22]:
def retrieve_similar_dreams(query_text, top_k=3):
    query_embedding = embedding_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = rag_index.search(query_embedding, top_k)

    results = []

    for score, index in zip(scores[0], indices[0]):
        results.append({
            "dream_id": dream_entries[index]["dream_id"],
            "similarity_score": float(score),
            "text": dream_entries[index]["text"]
        })

    return results

print("Retrieval function ready.")

Retrieval function ready.


In [23]:
test_dream = """
I was in a big city, it felt like Chicago. There were tall and wide brick buildings. This was like an isolated town, you'd drive
long desolate roads before suddenly reaching this burst of life in this pocket of an area. There were tons of food stands to the right
and I was so excited. But something felt off. Something felt scary...vulnerable. Eventually it got dark, and suddenly I was running
as fast as I could with the brick buildings to the left of me, with my direction heading back from the city and toward the entrance where
my car was parked. My car felt out of place and vulnerable. I felt like I had a lot of attention on me."""

retrieved_dreams = retrieve_similar_dreams(test_dream, top_k=3)

for result in retrieved_dreams:
    print(
        result["dream_id"],
        "- similarity:",
        round(result["similarity_score"], 3)
    )

18 - similarity: 0.664
19 - similarity: 0.57
dream11 - similarity: 0.532


## Unchanged Base Model

In [24]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    attn_implementation="eager"
)

print("Base model loaded.")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Base model loaded.


## Current Dream Only


In [25]:
def generate_baseline(dream_text):
    messages = [
        {
            "role": "user",
            "content": (
                "Analyze and summarize this dream. Identify its important people, places, "
                "events, emotions, atmosphere, prominent elements, and easily overlooked details. "
                "Then explore possible interpretations through past-life continuity, parallel "
                "realities, etheric or nonlocal information, energetic resonance, synchronicity, "
                "collective fields, field interference, temporal echoes, place memory, astral "
                "experience, and collective thought-forms. Connect each possibility to details "
                "actually present in the dream. Be deeply open-minded. "

                "\n\nDream:\n" + dream_text
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [27]:
def generate_prompt_v1(dream_text):
    messages = [
        {
            "role": "user",
            "content": (
                "Perform a deep, open-minded analysis of the dream below. "
                "Treat all metaphysical interpretations as speculative lenses. Summary at the end, brief. "
                "Do not invent, alter, or exaggerate any dream detail.\n\n"

                "Begin with SPECULATIVE LENSES so this section is never omitted. \n\n"

                "For every relevant lens:\n"
                "- Name the lens.\n"
                "- Explain how it could interpret the dream.\n"
                "- Quote or closely paraphrase the exact dream details supporting it.\n"
                "- Rate the feature match as strongly, moderately, or weakly activated.\n"
                "- Clearly state what information is missing or contradictory.\n\n"

                "Afterward, provide:\n"
                "1. A brief summary of what happened.\n"
                "2. The most prominent elements and why they are prominent.\n"
                "3. People, exact locations, and vague setting types.\n"
                "4. Important events, emotions, sensations, and atmosphere.\n"
                "5. Small or passing details that should not be forgotten.\n\n"

                "Use polished spelling and complete sentences. Separate stated dream facts "
                "from interpretations. "
                "Describe the writer as the title dreamer in the texts. Theory only.\n\n"

                "Dream:\n" + dream_text
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

prompt_v1_output = generate_prompt_v1(test_dream)

print(prompt_v1_output)

KeyboardInterrupt: 

### Prompt Version 2: Structured Analysis Without RAG

In [17]:
def generate_prompt_v2(dream_text):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an advanced dream-pattern analyst. "
                "Analyze dreams with deep curiosity and radical open-mindedness. "
                "Consider conventional explanations as well as speculative possibilities. "
                "Consider unconveitional perspectives involving consciousness, reality, time, "
                "memory, and connections between people, places, or other things. "
                "Focus only on possiblities genuinely related to the dream's specific details."
                "Explain what makes each possiblity worth considering without forcing connections."
                "Do not present speculation as fact"


                "Do not dismiss these possibilities merely because they are unconventional, "
                "but never present them as proven facts. Clearly distinguish dream evidence, "
                "possible interpretations, and ordinary alternative explanations. "
                "Do not make generic psychological claims about what the dreamer secretly wants, "
                "needs, fears, or is searching for in life."
            )
        },
        {
            "role": "user",
            "content": (
                "Analyze the following dream carefully and in depth.\n\n"
                f"DREAM:\n{dream_text}\n\n"

                "Follow this exact structure:\n\n"

                "1. DREAM RECORD\n"
                "Briefly reconstruct what happened using only information explicitly present "
                "in the dream. Do not add events or details.\n\n"

                "2. PROMINENT ELEMENTS\n"
                "Identify the strongest locations, people, objects, actions, atmosphere, "
                "emotions, bodily sensations, transitions, and unusual details. Explain which "
                "elements receive the most description or emotional intensity.\n\n"

                "3. SMALL DETAILS THAT SHOULD NOT BE LOST\n"
                "Preserve details mentioned only briefly that could become important when "
                "compared with future or previous dreams.\n\n"



                "For each selected lens:\n"
                "- Name the lens.\n"
                "- Identify the exact dream features that activated it.\n"
                "- Explain the possible relationship in detail.\n"
                "- State what additional evidence would strengthen it.\n"
                "- State what evidence would weaken or contradict it.\n"
                "- Give an ordinary process that could create a similar experience.\n"
                "- Suggest an observation that could help distinguish the possibilities.\n\n"

                "5. OTHER POSSIBLE LENSES\n"
                "Briefly identify any additional speculative, cultural, spiritual, cognitive, "
                "memory-based, or emotional frameworks that may be relevant.\n\n"

                "6. INTERFERENCE AND ALTERNATIVE EXPLANATIONS\n"
                "Consider memory incorporation, expectation, coincidence, false memory, source "
                "contamination, retrospective matching, recent experiences, sleep paralysis, "
                "ordinary dream construction, or other processes that could imitate an unusual "
                "connection. Do not automatically treat these explanations as superior; compare "
                "them fairly with the speculative possibilities.\n\n"

                "7. FINAL SYNTHESIS\n"
                "Summarize the dream's strongest patterns and most unusual relationships. "
                "State which possibilities appear most relevant and why, while clearly marking "
                "uncertainty. End with useful details or patterns to watch for in later dreams.\n\n"

                "Do not claim that any theory is proven. Do not diagnose the dreamer. "
                "Do not invent missing information. Be detailed, specific, and genuinely "
                "open-minded rather than giving a generic dream interpretation."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response


prompt_v2_output = generate_prompt_v2(test_dream)

print(prompt_v2_output)

1. **DREAM RECORD**
The individual recalls being in a large urban setting reminiscent of Chicago, surrounded by expansive brick structures indicative of skyscrapers or high rises. The environment initially feels secluded due to long stretches of empty roadways leading into densely populated areas filled with various eateries offering diverse cuisines. A sense of unease permeates throughout until darkness envelopes the scene, prompting frantic movement away from towering edifices towards their vehicle—a modern sedan seemingly incongruous within its surroundings. An overwhelming feeling of scrutiny accompanies every action taken during this nocturnal journey.

2. **PROMINENT ELEMENTS**
   - Urban Environment (Chicago): Activated by descriptions of "big city" ambiance coupled with distinct architectural landmarks such as "tall and wide brick buildings." Emotional Intensity revolves around feelings associated with isolation transitioning abruptly to bustling activity juxtaposed against vas

##Preliminary: Prompt Engineering

Objective

Evaluate how prompt structure affects Phi-3 Mini’s ability to preserve dream details, organize responses, and distinguish factual observations from speculative interpretations.

Experimental Setup

* Model: Phi-3 Mini 4K Instruct (4-bit quantization)
* Conditions: Baseline, Prompt V1, Prompt V2
* Controlled variables: Model, input dream, and generation settings
* Generation: Greedy decoding, 700 maximum tokens, repetition penalty of 1.1
* RAG / QLoRA: Disabled

Method

The baseline uses general dream-analysis instructions. Prompt V1 adds more detailed guidance for evaluating possible explanations, while Prompt V2 introduces explicit output sections and evidence-based reasoning.

Outputs will be compared for detail preservation, factual consistency, unsupported claims, and structural organization.

Purpose

This preliminary comparison tests prompt design before evaluating model fine-tuning in Arm 1.

In [25]:
def extract_dream_record(dream_text):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a factual information extraction system. "
                "Extract only information explicitly stated in the dream. "
                "Do not interpret, explain, infer, or invent information."
            )
        },
        {
            "role": "user",
            "content": f"""
Extract factual information from this dream.

Use EXACTLY these headings:

PEOPLE:
NAMED PLACES:
SETTING TYPES:
OBJECTS AND DETAILS:
EVENTS:
EMOTIONS AND SENSATIONS:
SEQUENCE:
UNCERTAINTIES:

Under each heading, use short bullet points.

RULES:
- Only include information supported by the dream.
- If a category has nothing, write "- none".
- Do not interpret symbolism.
- Do not explain your answers.
- Do not invent people, places, emotions, or events.
- NAMED PLACES means an explicitly identified location.
- Generic descriptions like "big city" belong under SETTING TYPES.
- If the dream says something "felt like" a place, do NOT claim it
  actually occurred there.
- Put comparisons or uncertain identifications under UNCERTAINTIES.
- Preserve small concrete details.
- Stop after UNCERTAINTIES.

DREAM:
{dream_text}
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [26]:
# Test the structured extractor on the existing city dream

test_record_raw = extract_dream_record(test_dream)

print("=" * 60)
print("STRUCTURED DREAM RECORD — TEST DREAM")
print("=" * 60)
print(test_record_raw)

STRUCTURED DREAM RECORD — TEST DREAM
PEOPLE:-none  

NAMELEDPLACES:-Chicago (implied)   

SETTTINGTYPES:-Big City/Urban Area; Isolated Towns within Big Cities    

OBJECTSDETAIS:-Food Stands; Brick Buildings; Car Entrance     

EVONTS:-Excitement initially turned into fearfulness & feeling exposed while escaping at nighttime      

SEQUESTRUE:-Walking through quiet streets leading up to lively spots then returning towards parking spot during dusk when cars feel unprotected       

UNDETCERTINSIZUES:-Something about being watched feels unusual but specific reasons aren’t clear - possibly due to personal feelings rather than actual circumstances described directly in the narrative


## Experimental Arm 1: QLoRA for Structured Dream Information Extraction

Research Question

Does QLoRA fine-tuning improve Phi-3 Mini’s accuracy and consistency when extracting structured information from dream narratives?

Hypothesis

Fine-tuning on manually labeled dream records will improve extraction completeness, factual accuracy, and category consistency while reducing unsupported details.

Dataset and Training

* Model: Phi-3 Mini 4K Instruct, 4-bit NF4 quantization
* Dataset: 24 dream entries; 18 training, 6 held-out testing
* Method: QLoRA using PEFT/LoRA
* Training: 3 epochs
* Output fields: PEOPLE, PLACES, SETTINGS, EVENTS, EMOTIONS, OBJECTS_DETAILS

The training examples pair each dream narrative with manually labeled structured records. The six test dreams were excluded from fine-tuning.

Experimental Conditions

* Condition A — Base: Original Phi-3 Mini without the LoRA adapter.
* Condition B — QLoRA: Same model with the trained adapter enabled.

Both conditions use the same six held-out dreams, extraction prompt, and generation settings.

Evaluation

Model outputs are manually compared against the original dream narratives using four criteria, each scored from 0–3:

1. Factual accuracy: Correctness of extracted information.
2. Completeness: Preservation of relevant dream details.
3. Unsupported detail avoidance: Absence of invented claims.
4. Structure: Correct classification into the six output fields.

The maximum score is 12 points per dream, or 72 points per model. Results are reported in the Arm 1 Results section.

In [30]:
# ------------------------------------------------------------
# ARM 1 — STRUCTURED EXTRACTION SCHEMA
# ------------------------------------------------------------

EXTRACTION_FIELDS = [
    "PEOPLE",
    "PLACES",
    "SETTINGS",
    "EVENTS",
    "EMOTIONS",
    "OBJECTS_DETAILS"
]

def format_reference(fields):
    return "\n".join(
        f"{field}: {', '.join(fields.get(field, [])) if fields.get(field) else 'none'}"
        for field in EXTRACTION_FIELDS
    )

print("Arm 1 extraction schema:")
for field in EXTRACTION_FIELDS:
    print("-", field)

Arm 1 extraction schema:
- PEOPLE
- PLACES
- SETTINGS
- EVENTS
- EMOTIONS
- OBJECTS_DETAILS


In [39]:

# ARM 1 — LOAD PRIVATE GOLD LABELS

import json

PRIVATE_LABEL_PATH = (
    "/content/drive/MyDrive/IT344/"
    "DJournal Project/private_labels/arm1_gold_labels.json"
)

with open(PRIVATE_LABEL_PATH, "r", encoding="utf-8") as f:
    gold_labels = json.load(f)

print("Private gold labels loaded successfully.")
print("Total labeled dreams:", len(gold_labels))


Private gold labels loaded successfully.
Total labeled dreams: 18


In [38]:

import os
import json

# Private storage location in Google Drive
PRIVATE_DIR = "/content/drive/MyDrive/IT344/DJournal Project/private_labels"
PRIVATE_LABEL_PATH = os.path.join(
    PRIVATE_DIR,
    "arm1_gold_labels.json"
)

# Create private folder if needed
os.makedirs(PRIVATE_DIR, exist_ok=True)

# Save the complete gold-label dictionary
with open(PRIVATE_LABEL_PATH, "w", encoding="utf-8") as f:
    json.dump(gold_labels, f, ensure_ascii=False, indent=2)

# Verify the saved file without revealing dream content
with open(PRIVATE_LABEL_PATH, "r", encoding="utf-8") as f:
    saved_labels = json.load(f)

assert saved_labels == gold_labels, "Save verification failed!"

print("Private gold labels saved and verified.")
print("Total labeled dreams:", len(saved_labels))
print("Dream content: not displayed.")


Private gold labels saved and verified.
Total labeled dreams: 18
Dream content: not displayed.


In [44]:
# ARM 1 — FIX TRAINING IDS

def normalize_id(value):
    return "dream" + str(value).lower().replace("dream", "").strip()

# Standardize all IDs
for dream in dream_entries:
    dream["dream_id"] = normalize_id(dream["dream_id"])

gold_labels = {
    normalize_id(k): v
    for k, v in gold_labels.items()
}

# Preserve the intended 18 training / 6 test split
train_ids = set(gold_labels.keys())

arm1_train_dreams = [
    d for d in dream_entries
    if d["dream_id"] in train_ids
]

arm1_test_dreams = [
    d for d in dream_entries
    if d["dream_id"] not in train_ids
]

assert len(arm1_train_dreams) == 18
assert len(arm1_test_dreams) == 6

print("Training:", len(arm1_train_dreams))
print("Testing:", len(arm1_test_dreams))
print("Gold labels matched successfully.")

Training: 18
Testing: 6
Gold labels matched successfully.


In [45]:
# ------------------------------------------------------------
# ARM 1 — BUILD QLORA TRAINING DATASET
# ------------------------------------------------------------

from datasets import Dataset

def format_gold_label(fields):
    return "\n".join([
        f"PEOPLE: {', '.join(fields['PEOPLE']) if fields['PEOPLE'] else 'none'}",
        f"PLACES: {', '.join(fields['PLACES']) if fields['PLACES'] else 'none'}",
        f"SETTINGS: {', '.join(fields['SETTINGS']) if fields['SETTINGS'] else 'none'}",
        f"EVENTS: {', '.join(fields['EVENTS']) if fields['EVENTS'] else 'none'}",
        f"EMOTIONS: {', '.join(fields['EMOTIONS']) if fields['EMOTIONS'] else 'none'}",
        f"OBJECTS_DETAILS: {', '.join(fields['OBJECTS_DETAILS']) if fields['OBJECTS_DETAILS'] else 'none'}"
    ])

training_rows = []

for dream in arm1_train_dreams:
    dream_id = dream["dream_id"]

    # Your files use 16/17/18 while the gold-label dictionary
    # uses dream16/dream17/dream18.
    label_id = dream_id
    if dream_id in {"16", "17", "18"}:
        label_id = f"dream{dream_id}"

    training_rows.append({
        "dream_id": dream_id,
        "dream_text": dream["text"],
        "target": format_gold_label(gold_labels[label_id])
    })

arm1_qlora_dataset = Dataset.from_list(training_rows)

assert len(arm1_qlora_dataset) == 18

print("=" * 60)
print("ARM 1 — QLORA TRAINING DATASET")
print("=" * 60)
print("Training examples:", len(arm1_qlora_dataset))
print("Columns:", arm1_qlora_dataset.column_names)
print("Dataset construction: PASSED")

ARM 1 — QLORA TRAINING DATASET
Training examples: 18
Columns: ['dream_id', 'dream_text', 'target']
Dataset construction: PASSED


In [46]:
# ------------------------------------------------------------
# ARM 1 — FORMAT QLORA TRAINING PROMPTS
# ------------------------------------------------------------

SYSTEM_PROMPT = """You are a structured dream information extractor.

Extract only information supported by the dream text.

Use exactly these six categories:
PEOPLE
PLACES
SETTINGS
EVENTS
EMOTIONS
OBJECTS_DETAILS

Rules:
- Do not interpret the dream.
- Do not invent details.
- Do not turn comparisons or uncertainty into confirmed facts.
- Preserve important names and concrete details.
- If a category has no supported information, write none.
"""

def make_training_text(example):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": "Extract the structured information from this dream:\n\n"
                       + example["dream_text"]
        },
        {
            "role": "assistant",
            "content": example["target"]
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

arm1_qlora_dataset = arm1_qlora_dataset.map(
    lambda example: {"text": make_training_text(example)}
)

assert len(arm1_qlora_dataset) == 18
assert "text" in arm1_qlora_dataset.column_names

print("=" * 60)
print("ARM 1 — TRAINING PROMPTS FORMATTED")
print("=" * 60)
print("Examples:", len(arm1_qlora_dataset))
print("Columns:", arm1_qlora_dataset.column_names)
print("Prompt formatting: PASSED")

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

ARM 1 — TRAINING PROMPTS FORMATTED
Examples: 18
Columns: ['dream_id', 'dream_text', 'target', 'text']
Prompt formatting: PASSED


In [47]:
# ------------------------------------------------------------
# ARM 1 — QLORA CONFIGURATION
# ------------------------------------------------------------

from peft import LoraConfig, prepare_model_for_kbit_training

# Prepare the existing 4-bit Phi-3 model for QLoRA.
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

qlora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "qkv_proj",
        "o_proj",
        "gate_up_proj",
        "down_proj"
    ]
)

print("=" * 60)
print("ARM 1 — QLORA CONFIGURATION")
print("=" * 60)
print("Base model:", MODEL_ID)
print("LoRA rank:", qlora_config.r)
print("LoRA alpha:", qlora_config.lora_alpha)
print("Target modules:", qlora_config.target_modules)
print("QLoRA configuration: PASSED")

ARM 1 — QLORA CONFIGURATION
Base model: microsoft/Phi-3-mini-4k-instruct
LoRA rank: 8
LoRA alpha: 16
Target modules: {'qkv_proj', 'o_proj', 'down_proj', 'gate_up_proj'}
QLoRA configuration: PASSED


In [48]:
# ------------------------------------------------------------
# ARM 1 — ATTACH QLORA ADAPTER
# ------------------------------------------------------------

from peft import get_peft_model

qlora_model = get_peft_model(base_model, qlora_config)

print("=" * 60)
print("ARM 1 — QLORA ADAPTER ATTACHED")
print("=" * 60)

qlora_model.print_trainable_parameters()

print("QLoRA adapter: PASSED")

ARM 1 — QLORA ADAPTER ATTACHED
trainable params: 12,582,912 || all params: 3,833,662,464 || trainable%: 0.3282
QLoRA adapter: PASSED


In [49]:
# ------------------------------------------------------------
# ARM 1 — QLORA TRAINER SETUP
# ------------------------------------------------------------

from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="/content/arm1_qlora_output",

    # Small 18-example experiment
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # QLoRA optimization
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    fp16=True,

    # Keep training efficient on T4
    max_length=1024,
    gradient_checkpointing=True,

    # Logging / saving
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",

    # Reproducibility
    seed=42,
)

trainer = SFTTrainer(
    model=qlora_model,
    args=training_args,
    train_dataset=arm1_qlora_dataset,
    processing_class=tokenizer,
)

print("=" * 60)
print("ARM 1 — QLORA TRAINER READY")
print("=" * 60)
print("Training examples:", len(arm1_qlora_dataset))
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Trainer setup: PASSED")

Adding EOS to train dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

ARM 1 — QLORA TRAINER READY
Training examples: 18
Epochs: 3
Learning rate: 0.0002
Trainer setup: PASSED


In [50]:
# ------------------------------------------------------------
# ARM 1 — FIX BF16 LORA PARAMETERS + TRAIN + SAVE
# ------------------------------------------------------------

import torch

# Cast ONLY trainable LoRA parameters to FP32.
# The 4-bit base model stays quantized.
for name, param in qlora_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Verify trainable parameter dtype
trainable_dtypes = {
    param.dtype
    for param in qlora_model.parameters()
    if param.requires_grad
}

print("Trainable parameter dtypes:", trainable_dtypes)

# Train
print("=" * 60)
print("ARM 1 — STARTING QLORA TRAINING")
print("=" * 60)

train_result = trainer.train()

print("=" * 60)
print("ARM 1 — QLORA TRAINING COMPLETE")
print("=" * 60)

print("Final training loss:", train_result.training_loss)

# Save permanently to Google Drive
SAVE_PATH = "/content/drive/MyDrive/IT344/DJournal Project/arm1_qlora_adapter"

trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Adapter saved permanently to:", SAVE_PATH)
print("Training: PASSED")

Trainable parameter dtypes: {torch.float32}
ARM 1 — STARTING QLORA TRAINING


Step,Training Loss
1,2.138391
2,1.970909
3,2.138928
4,2.025938
5,1.845407
6,1.910190
7,1.761644
8,1.677339
9,1.336739
10,1.460458


ARM 1 — QLORA TRAINING COMPLETE
Final training loss: 1.6833982865015666
Adapter saved permanently to: /content/drive/MyDrive/IT344/DJournal Project/arm1_qlora_adapter
Training: PASSED


In [52]:
# ------------------------------------------------------------
# ARM 1 — BUILD HELD-OUT TEST SET
# ------------------------------------------------------------

HELD_OUT_IDS = {
    "dream2",
    "dream8",
    "dream19",
    "dream20",
    "dream22",
    "dream23"
}

arm1_test_dreams = [
    dream for dream in dream_entries
    if dream["dream_id"] in HELD_OUT_IDS
]

found_ids = {dream["dream_id"] for dream in arm1_test_dreams}

assert found_ids == HELD_OUT_IDS, (
    f"Missing held-out dreams: {HELD_OUT_IDS - found_ids}"
)

assert not (found_ids & {normalize_id(x) for x in train_ids}), (
    "ERROR: A held-out dream is also in the training set."
)

print("=" * 60)
print("ARM 1 — HELD-OUT TEST SET")
print("=" * 60)
print("Held-out examples:", len(arm1_test_dreams))
print("IDs:", sorted(found_ids))
print("Training/test overlap: NONE")
print("Held-out test set: PASSED")

ARM 1 — HELD-OUT TEST SET
Held-out examples: 6
IDs: ['dream19', 'dream2', 'dream20', 'dream22', 'dream23', 'dream8']
Training/test overlap: NONE
Held-out test set: PASSED


In [55]:
train_used = {normalize_id(d["dream_id"]) for d in arm1_train_dreams}
test_used = {normalize_id(d["dream_id"]) for d in arm1_test_dreams}

print("Training dreams:", len(train_used))
print("Test dreams:", len(test_used))
print("Overlap:", train_used & test_used)

assert len(train_used) == 18
assert len(test_used) == 6
assert not (train_used & test_used)

print("Split verification: PASSED")

Training dreams: 18
Test dreams: 6
Overlap: set()
Split verification: PASSED


In [56]:
# ------------------------------------------------------------
# ARM 1 — QLORA OUTPUTS ON HELD-OUT DREAMS
# Store outputs silently for evaluation
# ------------------------------------------------------------

import torch

def generate_qlora_extraction(dream_text):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": "Extract the structured information from this dream:\n\n"
                       + dream_text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(qlora_model.device)

    qlora_model.eval()

    with torch.no_grad():
        outputs = qlora_model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


qlora_test_outputs = {}

for dream in arm1_test_dreams:
    dream_id = dream["dream_id"]

    result = generate_qlora_extraction(dream["text"])
    qlora_test_outputs[dream_id] = result

print("=" * 60)
print("ARM 1 — QLORA HELD-OUT GENERATION")
print("=" * 60)
print("Held-out dreams processed:", len(qlora_test_outputs))
print("Raw dream/output text stored.")
print("QLoRA held-out generation: PASSED")

ARM 1 — QLORA HELD-OUT GENERATION
Held-out dreams processed: 6
Raw dream/output text stored.
QLoRA held-out generation: PASSED


In [57]:
# ------------------------------------------------------------
# ARM 1 — BASE OUTPUTS WITH QLORA ADAPTER DISABLED
# ------------------------------------------------------------

import torch

def generate_base_extraction(dream_text):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content":
                "Extract the structured information from this dream:\n\n"
                + dream_text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(qlora_model.device)

    qlora_model.eval()

    # Temporarily disable LoRA = original base Phi-3 behavior
    with qlora_model.disable_adapter():
        with torch.no_grad():
            outputs = qlora_model.generate(
                **inputs,
                max_new_tokens=350,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


base_test_outputs = {}

for dream in arm1_test_dreams:
    dream_id = dream["dream_id"]
    base_test_outputs[dream_id] = generate_base_extraction(
        dream["text"]
    )

assert len(base_test_outputs) == 6

print("=" * 60)
print("ARM 1 — BASE HELD-OUT GENERATION")
print("=" * 60)
print("Held-out examples processed:", len(base_test_outputs))
print("Base held-out generation: PASSED")

ARM 1 — BASE HELD-OUT GENERATION
Held-out examples processed: 6
Base held-out generation: PASSED


In [58]:

# ARM 1 — SAVE BOTH MODEL OUTPUTS

import os
import json

SAVE_DIR = "/content/drive/MyDrive/IT344/DJournal Project/private_results"
os.makedirs(SAVE_DIR, exist_ok=True)

assert set(qlora_test_outputs) == set(base_test_outputs)

for filename, data in [
    ("qlora_outputs.json", qlora_test_outputs),
    ("base_outputs.json", base_test_outputs)
]:
    path = os.path.join(SAVE_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

print("Both model outputs saved successfully.")
print("Dreams compared:", len(qlora_test_outputs))


Both model outputs saved successfully.
Dreams compared: 6


In [3]:

import json
from pathlib import Path

folder = Path("/content/drive/MyDrive/IT344/DJournal Project/private_results")

with open(folder / "qlora_outputs.json", encoding="utf-8") as f:
    qlora_test_outputs = json.load(f)

with open(folder / "base_outputs.json", encoding="utf-8") as f:
    base_test_outputs = json.load(f)

assert len(qlora_test_outputs) == 6
assert set(qlora_test_outputs) == set(base_test_outputs)

print("Saved results loaded successfully.")
print("Dreams:", sorted(qlora_test_outputs))
print("Ready for evaluation: PASSED")


Saved results loaded successfully.
Dreams: ['dream19', 'dream2', 'dream20', 'dream22', 'dream23', 'dream8']
Ready for evaluation: PASSED


In [4]:
print("QLoRA output type:", type(qlora_test_outputs).__name__)
print("Base output type:", type(base_test_outputs).__name__)

first_id = sorted(qlora_test_outputs)[0]

print("First dream ID:", first_id)
print("QLoRA entry type:", type(qlora_test_outputs[first_id]).__name__)
print("Base entry type:", type(base_test_outputs[first_id]).__name__)

QLoRA output type: dict
Base output type: dict
First dream ID: dream19
QLoRA entry type: str
Base entry type: str


In [5]:
import pandas as pd

comparison = pd.DataFrame({
    "Dream ID": sorted(qlora_test_outputs),
    "Base Output": [
        base_test_outputs[d] for d in sorted(qlora_test_outputs)
    ],
    "QLoRA Output": [
        qlora_test_outputs[d] for d in sorted(qlora_test_outputs)
    ]
})

print("ARM 1 — BASE VS QLORA COMPARISON")
print("Dreams:", len(comparison))
display(comparison[["Dream ID"]])

ARM 1 — BASE VS QLORA COMPARISON
Dreams: 6


,Dream ID
0,dream19
1,dream2
2,dream20
3,dream22
4,dream23
5,dream8


In [6]:
length_check = comparison[["Dream ID"]].copy()

length_check["Base characters"] = comparison["Base Output"].str.len()
length_check["QLoRA characters"] = comparison["QLoRA Output"].str.len()

display(length_check)

,Dream ID,Base characters,QLoRA characters
0,dream19,391,590
1,dream2,238,366
2,dream20,294,550
3,dream22,167,436
4,dream23,217,472
5,dream8,153,554


In [9]:

# ARM 1 — HUMAN EVALUATION: BASE VS QLORA
# Scores are manual judgments against the original six dreams.
# Criteria: accuracy, completeness, unsupported-detail avoidance, structure.
# Scale: 0 = poor, 1 = weak, 2 = mostly correct, 3 = strong.

import pandas as pd
import json
from pathlib import Path

criteria = [
    "Accuracy",
    "Completeness",
    "Unsupported detail avoidance",
    "Structure"
]

# Scores are ordered according to the four criteria above.
human_scores = {
    "dream2":  {"Base": [3, 1, 3, 2], "QLoRA": [3, 2, 2, 2]},
    "dream8":  {"Base": [3, 1, 3, 2], "QLoRA": [3, 3, 2, 2]},
    "dream19": {"Base": [3, 2, 3, 2], "QLoRA": [2, 3, 1, 2]},
    "dream20": {"Base": [3, 2, 3, 1], "QLoRA": [2, 2, 2, 2]},
    "dream22": {"Base": [3, 2, 3, 1], "QLoRA": [1, 2, 0, 2]},
    "dream23": {"Base": [2, 3, 2, 3], "QLoRA": [1, 2, 1, 2]}
}

assert set(human_scores) == set(base_test_outputs)
assert set(human_scores) == set(qlora_test_outputs)

rows = []

for dream_id, models in human_scores.items():
    for model, scores in models.items():
        assert len(scores) == 4
        assert all(isinstance(s, int) and 0 <= s <= 3 for s in scores)

        row = {"Dream ID": dream_id, "Model": model}
        row.update(dict(zip(criteria, scores)))
        row["Total"] = sum(scores)
        rows.append(row)

evaluation = pd.DataFrame(rows)

results = evaluation.pivot(
    index="Dream ID",
    columns="Model",
    values="Total"
)[["Base", "QLoRA"]]

results["Difference (QLoRA - Base)"] = (
    results["QLoRA"] - results["Base"]
)

print("ARM 1 — HUMAN EVALUATION (MAX 12 PER DREAM)")
display(results)

summary = evaluation.groupby("Model")["Total"].agg(
    ["sum", "mean"]
)

summary["Percent"] = summary["sum"] / 72 * 100

print("OVERALL RESULTS")
display(summary.round(2))

save_dir = Path(
    "/content/drive/MyDrive/IT344/DJournal Project/private_results"
)
save_dir.mkdir(parents=True, exist_ok=True)

evaluation.to_csv(
    save_dir / "arm1_human_evaluation.csv",
    index=False
)

with open(save_dir / "arm1_human_evaluation.json", "w") as f:
    json.dump(human_scores, f, indent=2)

print("Evaluation saved.")


ARM 1 — HUMAN EVALUATION (MAX 12 PER DREAM)


Model,Base,QLoRA,Difference (QLoRA - Base)
Dream ID,,,
dream19,10,8,-2
dream2,9,9,0
dream20,9,8,-1
dream22,9,5,-4
dream23,10,6,-4
dream8,9,10,1


OVERALL RESULTS


,sum,mean,Percent
Model,,,
Base,56,9.33,77.78
QLoRA,46,7.67,63.89


Evaluation saved.



### Arm 1 Results

The base Phi-3 Mini model and QLoRA-adapted model were evaluated on the same six held-out dreams. Each output was manually scored for factual accuracy, completeness, avoidance of unsupported details, and extraction structure, using a 0–3 scale per criterion.

The base model received 56/72 points (77.8%), while the QLoRA model received 46/72 points (63.9%).

QLoRA extracted additional events and details in some cases, but its responses also included unsupported claims and category errors.

These results do not support the original hypothesis that QLoRA would improve extraction quality. However, the evaluation was limited to six held-out dreams, a small training dataset, and subjective human scoring. Further training and testing would be needed to determine whether the result generalizes.
